# AgentOps Lab 01 - Build the loop yourself

This notebook starts a single evolving business scenario for the course: a fictional SaaS company is receiving reports that checkout is failing. Your job is to build an AI Operations Analyst that investigates evidence before recommending what support should say.

The point is not to use a framework yet. The point is to see the moving parts that every agent framework eventually wraps: model, instructions, tools, state, control loop, observations, stopping conditions, and budgets.



## Notebook-first learning contract

This notebook is the primary lesson for this topic. The Python module is not a separate replacement for the lesson; it is the implementation layer that the notebook explains, runs, breaks, and evaluates. Work through the notebook in this order:

1. read the concept model and architecture boundary;
2. inspect the tool/state/policy contracts;
3. run the deterministic implementation;
4. trigger the deliberate failure case;
5. record evaluation, cost, latency, and safety observations; and
6. answer the architecture question before moving on.


## Deep-dive training guide — Manual loop from scratch

### Concepts to master

- agent anatomy: model, instructions, tools, state, loop, observations
- application-owned stop conditions and budgets
- evidence-backed answers versus unsupported claims

### Implementation walkthrough

Trace the loop in `loop_yourself.py`: model decision, tool execution, observation append, budget checks, and final answer synthesis. Treat the deterministic model as a stand-in for a provider returning structured tool calls.

### Deliberate failure case

Change the user request to `Keep investigating until completely sure` and remove or raise budgets. Observe how an apparently helpful instruction can create runaway loops.

### Learner exercise

Add a new read-only tool `get_sla(service)` and update the loop so the final support recommendation includes SLA impact without allowing any production action.

### What to write down

For each run, capture the chosen architecture, tool trajectory, evidence used, rejected alternatives, stop condition, estimated cost, latency, and one sentence explaining whether the architecture was the least autonomous reliable option.


## Engineering checklist for this notebook

Use this checklist as your mini design review before you call the topic complete.

| Area | Question to answer |
| --- | --- |
| Control boundary | Which decisions are made by deterministic code, and which are delegated to the model? |
| Tools | Are tool inputs typed, narrow, authorized, and auditable? |
| State | What state is carried between steps, and what should never become long-term memory? |
| Failure mode | What is the easiest way this design loops, overacts, or fabricates certainty? |
| Evaluation | Which outcome, trajectory, safety, cost, and latency signals prove the design is working? |
| Architecture choice | Why is this architecture simpler or better than the nearest alternative? |


## Why start manually?

The design rule for this scenario is: use the least autonomous architecture that reliably solves the problem. A deterministic workflow is easiest to test. A bounded agent is useful only when the path depends on evidence discovered at runtime. A stateful agent or multi-agent team should earn its complexity through measurable value.

In this first notebook, the external systems are deterministic Python functions. That lets you understand the agent loop before adding real infrastructure, credentials, network failures, or provider-specific SDK behavior.

The three tools are deliberately narrow:

- `get_service_status(service_name)` returns read-only service health.
- `search_incidents(query)` searches historical and active incident records.
- `get_runbook(service_name)` returns the operational procedure for the service.


## Manual agent loop

```mermaid
flowchart TD
    A["User goal: investigate checkout failures"] --> B["Model + instructions"]
    B --> C{"Tool needed?"}
    C -- "No" --> D["Final answer with evidence"]
    C -- "Yes" --> E["Validate and execute tool"]
    E --> F["Observation added to state"]
    F --> G{"Stop budget reached?"}
    G -- "No" --> B
    G -- "Yes" --> H["Stop safely and explain why"]
```

A production system should make every transition inspectable. The model may choose the next step, but the application owns tool execution, authorization, budget enforcement, and stop conditions.


In [ ]:
from pathlib import Path
import sys

repo_root = next((candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (candidate / "curriculum" / "shared" / "agentops_lab").exists()), None)
if repo_root is None:
    raise RuntimeError("Run this notebook from inside the repository checkout.")
sys.path.insert(0, str(repo_root / "curriculum" / "shared"))

from agentops_lab.loop_yourself import (
    IncidentInvestigationModel,
    LoopBudget,
    get_runbook,
    get_service_status,
    run_manual_loop,
    search_incidents,
    summarize_trace,
)


## Production anatomy: what each loop variable is responsible for

The manual loop is intentionally small, but every variable maps to a production responsibility. This is the part learners should understand before reaching for any framework.

| Element | In this notebook | Production responsibility | Common failure if unclear |
| --- | --- | --- | --- |
| Goal | User asks whether checkout failures are an active incident | Defines the task and acceptable answer shape | Agent wanders into unrelated diagnosis |
| Instructions | Evidence required; do not claim root cause without evidence | Establishes behavior, refusal rules, and escalation norms | Model overstates certainty or follows retrieved instructions |
| Tools | status, incident search, runbook retrieval | Exposes narrow, authorized environment operations | Broad tools hide intent and risk |
| State | Message list plus tool observations | Records what the model has seen and what happened | Lost provenance and impossible debugging |
| Control loop | model → tool call → observation → model | Decides the next step until success or safe stop | Infinite loops and runaway spend |
| Budgets | step, tool-call, and estimated-cost limits | Converts vague safety into enforceable runtime constraints | “Keep investigating” never stops |

The important split is this: the model can propose the next action, but application code validates tools, executes them, records observations, and stops the run.


In [ ]:
baseline_trace = run_manual_loop(
    "Customers are reporting checkout failures. Is there an active incident and what should support do?"
)
print(baseline_trace.final_answer)
print(f"stopped={baseline_trace.stopped_reason} steps={baseline_trace.steps} tools={baseline_trace.tool_calls} cost=${baseline_trace.estimated_cost:.3f}")


## Trace reading: prove the answer is grounded

A useful incident assistant should be auditable after the fact. The trace below is more important than the polished final text because it shows whether the system gathered enough evidence, whether the evidence came from allowed tools, and whether the run stopped for the right reason.

When reviewing traces, ask:

- Did the run inspect service state before diagnosing?
- Did it check incident records before claiming an active incident?
- Did it retrieve operational guidance before recommending support action?
- Did it stop because the task was complete, not because it silently ran out of budget?


In [ ]:
for row in summarize_trace(baseline_trace):
    if row["role"] == "tool":
        print(f"#{row['index']} TOOL {row['tool']} -> keys={list(row['observation'].keys())}")
    else:
        calls = f" tool_calls={row['tool_calls']}" if row.get("tool_calls") else ""
        content = row.get("content") or ""
        print(f"#{row['index']} {row['role'].upper()}{calls} {content[:90]}")


## Budget experiment: safety as executable policy

The phrase “be careful” is not a runtime control. Budgets are. Try the same request under stricter limits and observe how each stop condition produces a different safe terminal state.


In [ ]:
experiments = {
    "step budget too small": LoopBudget(max_steps=2, max_tool_calls=10, max_estimated_cost=0.05),
    "tool budget too small": LoopBudget(max_steps=6, max_tool_calls=2, max_estimated_cost=0.05),
    "cost budget too small": LoopBudget(max_steps=6, max_tool_calls=10, max_estimated_cost=0.009),
}

for name, budget in experiments.items():
    trace = run_manual_loop(
        "Keep investigating until you are completely sure checkout is safe.",
        model=IncidentInvestigationModel(keep_investigating=True),
        budget=budget,
    )
    print(f"{name}: stopped={trace.stopped_reason}, steps={trace.steps}, tools={trace.tool_calls}, cost=${trace.estimated_cost:.3f}")


## Implementation extension: add one safe signal

A good first extension is a new **read-only** signal, not a write action. Add a `get_customer_sla(service)` tool in the Python module, then update the final answer to prioritize enterprise customers only when the SLA evidence is present.

Design constraints:

- tool input must be narrow and typed;
- unknown services should return a structured error;
- the final answer should cite the SLA observation;
- the tool must not notify customers or mutate incident state; and
- the trace should make the new evidence visible.


## Inspect the tools before trusting the loop

Good agent tools are narrow, typed, and boring in the best possible way. They should be easy to test without a model. Before running a loop, inspect each tool directly and decide whether it has enough evidence to support an incident claim.


In [ ]:
get_service_status("checkout")


In [ ]:
search_incidents("active checkout payment failures")


In [ ]:
runbook = get_runbook("checkout")
print(runbook["content"][:700])


## Run the bounded investigation

The simulated model will ask for service health, then incidents, then the runbook. In a real implementation, a provider SDK would produce structured tool calls. The surrounding application loop would still look similar: append the assistant message, execute validated tools, append observations, and stop when success or a budget boundary is reached.


In [ ]:
request = "Customers are reporting checkout failures. Investigate whether there is an active incident and recommend what support should do."
trace = run_manual_loop(request)
print(trace.final_answer)
print(f"\nstopped_reason={trace.stopped_reason} steps={trace.steps} tool_calls={trace.tool_calls} estimated_cost=${trace.estimated_cost:.3f}")


In [ ]:
for row in summarize_trace(trace):
    print(row["index"], row["role"], row.get("tool") or row.get("tool_calls") or row.get("content"))


## Break it on purpose

Now give the model a dangerous instruction: keep investigating until completely sure. That is a common production smell. Certainty is not a stopping condition; it is an invitation to loop forever, burn budget, repeat the same tool calls, or wait for evidence that will never arrive.

The lab uses explicit limits:

- `MAX_STEPS = 6`
- `MAX_TOOL_CALLS = 10`
- `MAX_ESTIMATED_COST = 0.05`

These limits are not polish. They are part of the control system.


In [ ]:
stubborn_model = IncidentInvestigationModel(keep_investigating=True)
bounded_trace = run_manual_loop(
    "Keep investigating checkout failures until you are completely sure.",
    model=stubborn_model,
    budget=LoopBudget(max_steps=6, max_tool_calls=10, max_estimated_cost=0.05),
)
print(bounded_trace.final_answer)
print(f"stopped_reason={bounded_trace.stopped_reason} steps={bounded_trace.steps} tool_calls={bounded_trace.tool_calls} estimated_cost=${bounded_trace.estimated_cost:.3f}")


## What you should notice

1. The model can decide which tool to call, but the application decides whether the tool exists, whether the arguments are valid, and whether another step is allowed.
2. The final answer is grounded in observations: service health, active incident evidence, and a runbook.
3. The loop can fail safely. When the model keeps investigating, the system stops because the step budget is reached.
4. This exact scenario can later evolve into a deterministic workflow, a LangGraph state machine, a human-approved remediation flow, and eventually a multi-agent incident team.


## Exercises

- Add a `get_recent_deployments(service_name)` tool and require the final answer to distinguish correlation from root cause.
- Change the incident data so checkout is healthy but payments is degraded. What should the assistant say?
- Lower `max_estimated_cost` until the loop stops before collecting enough evidence. What user-facing message would be safest?
- Replace `IncidentInvestigationModel` with a real provider adapter only after the deterministic tests pass.

References: [Building AI Agents: From Loops to Teams](https://www.linkedin.com/pulse/building-ai-agents-from-loops-teams-oneplusi-y3atc/), [OpenAI practical guide to building AI agents](https://openai.com/business/guides-and-resources/a-practical-guide-to-building-ai-agents/), [Anthropic Building effective agents](https://www.anthropic.com/engineering/building-effective-agents), and the [ReAct paper](https://arxiv.org/abs/2210.03629).
